# 01 — Exploratory Data Analysis (EDA)

**Project:** AI-Powered SOC — modular ML engine (independent from live SIEM)

**Existing operational SOC (do not modify in this notebook):**
- Wazuh 4.14.9 + Suricata 8.0.7
- SOC / Manager / Indexer / Dashboard: `192.168.221.138`
- IDS / Suricata: `192.168.221.136`
- Kali (lab attacker): `192.168.221.30`

**Dataset for this notebook:** CIC-IDS2017 traffic-flow CSVs (ISCX feature set).  
**Important:** CIC flow features are **not** identical to Wazuh/Suricata alert fields. This EDA builds understanding of a research dataset. Future FastAPI integration must map SIEM events into a compatible feature schema (documented at the end).

**Goals**
1. Load and profile the raw CSVs reproducibly
2. Analyze missing / infinite values, duplicates, constant columns, class imbalance
3. Visualize attack distributions, numerical features, and correlations
4. Study feature–label relationships
5. Identify leakage / columns unsafe for training
6. Produce clear conclusions for the next modeling stage


## 0. Environment & reproducibility

Configures paths, plotting defaults, and a safe loader that searches several candidate directories. If CSVs are missing, later analysis cells stop cleanly with instructions (**no fabricated statistics**).


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
    HAS_SNS = True
except Exception:
    HAS_SNS = False
    print("seaborn not available — using matplotlib only")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

NOTEBOOK_DIR = Path.cwd().resolve()
CANDIDATE_RAW_DIRS = [
    NOTEBOOK_DIR / "data" / "raw",
    NOTEBOOK_DIR / ".." / "data" / "raw",
    NOTEBOOK_DIR / ".." / ".." / "data" / "raw",
    Path("../data/raw"),
    Path("../../data/raw"),
    Path("data/raw"),
]

OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

print("cwd:", NOTEBOOK_DIR)
print("seaborn:", HAS_SNS)
print("output_dir:", OUTPUT_DIR.resolve())


## 1. Data loading

Preserves the original concatenation approach from `01_eda.ipynb`, with multiple search paths and graceful failure if files are absent.


In [ ]:
def find_raw_dir(candidates=CANDIDATE_RAW_DIRS):
    for d in candidates:
        d = d.resolve()
        if d.is_dir() and any(d.glob("*.csv")):
            return d
    return None


def load_cic_raw(raw_dir: Path) -> pd.DataFrame:
    files = sorted(raw_dir.glob("*.csv"))
    print(len(files), "fichiers")
    dfs = []
    for f in files:
        d = pd.read_csv(f, low_memory=False)
        d.columns = d.columns.str.strip()
        d["source_file"] = f.name
        dfs.append(d)
        print(f.name, d.shape)
    if not dfs:
        raise FileNotFoundError(f"No CSV files in {raw_dir}")
    return pd.concat(dfs, ignore_index=True)


RAW_DIR = find_raw_dir()
DATA_LOADED = False
df = None

if RAW_DIR is None:
    print("ERROR: CIC-IDS2017 CSV directory not found.")
    print("Place the ISCX CSV files under one of:")
    for d in CANDIDATE_RAW_DIRS:
        print(" -", d.resolve())
    print("Expected files include names like Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
else:
    print("Using raw dir:", RAW_DIR)
    df = load_cic_raw(RAW_DIR)
    DATA_LOADED = True
    print("combined shape:", df.shape)
    if "Label" in df.columns:
        print(df["Label"].value_counts())
    else:
        print("WARNING: 'Label' column missing. Columns sample:", list(df.columns)[:30])


### Loading notes (from original successful run)

When the full CIC set was available, `01_eda.ipynb` loaded **8 CSV files** totaling **2,830,743 rows × 80 columns** (features + `source_file`).

Observed label counts in that run:

| Label | Count |
|-------|------:|
| BENIGN | 2,273,097 |
| DoS Hulk | 231,073 |
| PortScan | 158,930 |
| DDoS | 128,027 |
| DoS GoldenEye | 10,293 |
| FTP-Patator | 7,938 |
| SSH-Patator | 5,897 |
| DoS slowloris | 5,796 |
| DoS Slowhttptest | 5,499 |
| Bot | 1,966 |
| Web Attack Brute Force | 1,507 |
| Web Attack XSS | 652 |
| Infiltration | 36 |
| Web Attack Sql Injection | 21 |
| Heartbleed | 11 |

If CSVs are missing in this environment, fix `data/raw` and re-run — do not invent new statistics beyond what cells compute.


## 2. Label cleaning & data-quality audit

Preserves the original cleaning / Inf / NaN / duplicate / constant-column / imbalance checks with clearer reporting.


In [ ]:
if not DATA_LOADED:
    raise SystemExit("Dataset not loaded — fix RAW_DIR and re-run from section 1.")

# Label normalization (original intent preserved)
df["Label"] = (
    df["Label"].astype(str)
    .str.replace("\ufffd", "-", regex=False)
    .str.replace("Web Attack - ", "Web Attack ", regex=False)
    .str.strip()
)

df["is_attack"] = (df["Label"] != "BENIGN").astype(int)

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_num_cols = [c for c in num_cols if c != "is_attack"]

num = df[feature_num_cols]
inf_per_col = np.isinf(num).sum()
inf_total = int(inf_per_col.sum())
nan_per_col = df.isna().sum()
nan_total = int(nan_per_col.sum())

print("Inf total:", inf_total)
print("NaN total:", nan_total)
print("\nColumns with NaN:")
print(nan_per_col[nan_per_col > 0])
print("\nColumns with Inf:")
print(inf_per_col[inf_per_col > 0])

feat_cols = [c for c in df.columns if c != "source_file"]
dup_count = int(df.duplicated(subset=feat_cols).sum())
print(f"\nDoublons : {dup_count} ({dup_count / len(df):.1%})")

const_cols = [c for c in feature_num_cols if df[c].nunique(dropna=False) <= 1]
print("Colonnes constantes :", const_cols)

label_counts = df["Label"].value_counts()
label_pct = (df["Label"].value_counts(normalize=True) * 100).round(3)
print("\nLabel distribution (%):")
print(label_pct)

binary_pct = (df["is_attack"].value_counts(normalize=True) * 100).round(3)
print("\nBinary benign vs attack (%):")
print(binary_pct.rename({0: "BENIGN", 1: "ATTACK"}))

quality = {
    "n_rows": int(len(df)),
    "n_cols": int(df.shape[1]),
    "inf_total": inf_total,
    "nan_total": nan_total,
    "duplicate_rows": dup_count,
    "duplicate_pct": float(dup_count / len(df)),
    "constant_columns": const_cols,
    "label_counts": {str(k): int(v) for k, v in label_counts.items()},
}
(OUTPUT_DIR / "quality_summary.json").write_text(json.dumps(quality, indent=2), encoding="utf-8")
print("\nWrote", OUTPUT_DIR / "quality_summary.json")


### Data-quality conclusions

From the original executed notebook (reconfirmed when data is present):

| Issue | Observation | Implication |
|-------|-------------|-------------|
| Missing values | Primarily `Flow Bytes/s` (≈1,358 NaNs) | Impute or carefully drop before training |
| Infinite values | `Flow Bytes/s` (~1,509) and `Flow Packets/s` (~2,867) | Replace Inf→NaN then impute/clip; zero-duration flows break rates |
| Duplicates | ≈**10.9%** duplicate feature rows | Drop before split to avoid train/test leakage via clones |
| Constant columns | `Bwd PSH Flags`, `Bwd URG Flags`, bulk-average fields | Drop — no predictive value |
| Class imbalance | **~80.3% BENIGN**; rare classes ≪ 0.1% | Stratified splits + class weights / resampling; prefer F1/PR over accuracy |

**SOC note:** Strong benign majority mirrors real traffic. Optimize for minority attack detection.


## 3. Attack distribution visualizations


In [ ]:
if not DATA_LOADED:
    raise SystemExit("Dataset not loaded.")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = label_counts.index.tolist()
axes[0].barh(order[::-1], label_counts.loc[order[::-1]].values, color="#2c7fb8")
axes[0].set_xscale("log")
axes[0].set_xlabel("Count (log scale)")
axes[0].set_title("CIC-IDS2017 — Label counts")

bin_counts = df["is_attack"].value_counts().rename({0: "BENIGN", 1: "ATTACK"})
axes[1].bar(bin_counts.index.astype(str), bin_counts.values, color=["#99d8c9", "#e34a33"])
axes[1].set_ylabel("Count")
axes[1].set_title("Binary view: Benign vs Attack")
for i, v in enumerate(bin_counts.values):
    axes[1].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "label_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

src = (
    df.groupby("source_file")["is_attack"]
    .agg(total="count", attacks="sum")
    .assign(attack_rate=lambda x: x["attacks"] / x["total"])
    .sort_values("attack_rate", ascending=False)
)
print(src)
src.to_csv(OUTPUT_DIR / "attack_rate_by_source_file.csv")


### Distribution conclusions

- **BENIGN** dominates (~80%).
- High-volume attacks in the original run: **DoS Hulk**, **PortScan**, **DDoS**.
- Rare labels (Web attacks, Infiltration, Heartbleed) need grouping or specialized handling.
- `source_file` reflects which attack day was simulated → **metadata only**, not a training feature.


## 4. Numerical feature profiling

Descriptive statistics and distributions for key volume/rate/timing features (Inf removed for plotting).


In [ ]:
if not DATA_LOADED:
    raise SystemExit("Dataset not loaded.")

plot_df = df.copy()
plot_df[feature_num_cols] = plot_df[feature_num_cols].replace([np.inf, -np.inf], np.nan)

desc = plot_df[feature_num_cols].describe().T
desc["missing"] = plot_df[feature_num_cols].isna().sum()
desc["missing_pct"] = (desc["missing"] / len(plot_df) * 100).round(4)
desc_path = OUTPUT_DIR / "numeric_describe.csv"
desc.to_csv(desc_path)
print("Wrote", desc_path)
print(desc.sort_values("missing_pct", ascending=False).head(15))

candidate_feats = [
    "Flow Duration", "Flow Bytes/s", "Flow Packets/s",
    "Total Fwd Packets", "Total Backward Packets",
    "Fwd Packet Length Mean", "Bwd Packet Length Mean", "Packet Length Mean",
    "Average Packet Size", "Fwd IAT Mean", "Bwd IAT Mean", "Active Mean", "Idle Mean",
]
present_feats = [c for c in candidate_feats if c in plot_df.columns]
print("Plotting features:", present_feats)

n = len(present_feats)
cols = 3
rows = int(np.ceil(max(n, 1) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, present_feats):
    s = plot_df[col].dropna()
    lo, hi = s.quantile([0.01, 0.99])
    s2 = s[(s >= lo) & (s <= hi)]
    ax.hist(s2, bins=50, color="#6baed6", edgecolor="white")
    ax.set_title(col)
for ax in axes[len(present_feats):]:
    ax.axis("off")
plt.suptitle("Numerical feature distributions (1st–99th percentile window)", y=1.01)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "numeric_histograms.png", dpi=150, bbox_inches="tight")
plt.show()


### Numerical feature conclusions

- Many features are **right-skewed** → prefer robust scaling / log1p in modeling.
- Rate features break on near-zero duration → clean Inf/NaN before training.
- Drop constant bulk-related columns early.


## 5. Correlations among numerical features

Pearson correlation after Inf→NaN. Large frames are optionally sampled with a fixed seed for speed.


In [ ]:
if not DATA_LOADED:
    raise SystemExit("Dataset not loaded.")

usable = [c for c in feature_num_cols if c not in const_cols]
corr_base = plot_df[usable].dropna(axis=1, how="all")

MAX_CORR_ROWS = 150_000
if len(corr_base) > MAX_CORR_ROWS:
    corr_sample = corr_base.sample(MAX_CORR_ROWS, random_state=RANDOM_STATE)
    print(f"Correlation sample: {MAX_CORR_ROWS} rows of {len(corr_base)}")
else:
    corr_sample = corr_base

corr = corr_sample.corr(method="pearson")
corr.to_csv(OUTPUT_DIR / "feature_correlation.csv")

variances = corr_sample.var().sort_values(ascending=False)
top_feats = variances.head(25).index.tolist()
corr_top = corr.loc[top_feats, top_feats]

fig, ax = plt.subplots(figsize=(12, 10))
if HAS_SNS:
    sns.heatmap(corr_top, cmap=0.8, cmap="vlag", center=0, ax=ax)
else:
    im = ax.imshow(corr_top.values, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(top_feats)))
    ax.set_yticks(range(len(top_feats)))
    ax.set_xticklabels(top_feats, rotation=90, fontsize=8)
    ax.set_yticklabels(top_feats, fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("Pearson correlation — top-variance features")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "correlation_heatmap_top25.png", dpi=150, bbox_inches="tight")
plt.show()

pairs = []
cols_ = corr.columns
for i in range(len(cols_)):
    for j in range(i + 1, len(cols_)):
        r = corr.iloc[i, j]
        if abs(r) >= 0.95:
            pairs.append((cols_[i], cols_[j], float(r)))
pairs = sorted(pairs, key=lambda x: -abs(x[2]))
print(f"Highly correlated pairs (|r|>=0.95): {len(pairs)}")
for a, b, r in pairs[:30]:
    print(f"  {a}  <->  {b}  r={r:.4f}")


### Correlation conclusions

- CIC exports include near-redundant length/IAT statistics; high |r| pairs are drop candidates.
- Confirm with model importance before aggressive removal.


## 6. Relationships between features and attack labels


In [ ]:
if not DATA_LOADED:
    raise SystemExit("Dataset not loaded.")

rel_feats = [c for c in [
    "Flow Duration", "Flow Bytes/s", "Flow Packets/s",
    "Total Fwd Packets", "Packet Length Mean", "Fwd IAT Mean", "Idle Mean",
] if c in plot_df.columns]

med = plot_df.groupby("is_attack")[rel_feats].median().T
med.columns = ["median_benign", "median_attack"]
med["ratio_attack_over_benign"] = med["median_attack"] / med["median_benign"].replace(0, np.nan)
print(med)
med.to_csv(OUTPUT_DIR / "feature_medians_by_binary_label.csv")

sample = plot_df.sample(min(80_000, len(plot_df)), random_state=RANDOM_STATE)
fig, axes = plt.subplots(len(rel_feats), 1, figsize=(10, 3.2 * max(len(rel_feats), 1)))
if len(rel_feats) == 1:
    axes = [axes]
for ax, col in zip(axes, rel_feats):
    s = sample[[col, "is_attack"]].replace([np.inf, -np.inf], np.nan).dropna()
    lo, hi = s[col].quantile([0.01, 0.99])
    s = s[(s[col] >= lo) & (s[col] <= hi)]
    data = [s.loc[s["is_attack"] == 0, col], s.loc[s["is_attack"] == 1, col]]
    ax.boxplot(data, labels=["BENIGN", "ATTACK"], showfliers=False)
    ax.set_title(f"{col} by class (1–99% window)")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "feature_vs_label_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

if "Flow Duration" in plot_df.columns:
    top_labels = label_counts.head(8).index
    sub = plot_df[plot_df["Label"].isin(top_labels)]
    means = sub.groupby("Label")["Flow Duration"].median().reindex(top_labels)
    fig, ax = plt.subplots(figsize=(10, 4))
    means.plot(kind="bar", ax=ax, color="#fc8d59")
    ax.set_ylabel("Median Flow Duration")
    ax.set_title("Median Flow Duration by top labels")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "flow_duration_by_label.png", dpi=150, bbox_inches="tight")
    plt.show()


### Feature–label conclusions

- Duration / rate / size features often separate attacks from benign traffic.
- Attack families differ (PortScan ≠ DoS Hulk) → consider binary detection then family classification.
- Validate generalization across days (`source_file` blocking), not only random splits.


## 7. Data leakage & columns to exclude from training


In [ ]:
if not DATA_LOADED:
    raise SystemExit("Dataset not loaded.")

exclude = {
    "Label": "Target label — not an input feature",
    "is_attack": "Derived from Label — leakage if used as input",
    "source_file": "Day/attack-script provenance; tied to label generation",
}

possible_id_like = [
    "Flow ID", "Source IP", "Destination IP", "Source Port", "Destination Port",
    "Protocol", "Timestamp", "Src IP", "Dst IP", "Src Port", "Dst Port",
]
for c in possible_id_like:
    if c in df.columns:
        exclude[c] = "Identifier / environment-specific metadata"

for c in const_cols:
    exclude[c] = "Constant column — no information"

optional_careful = {}
for c, reason in {
    "Flow Bytes/s": "Contains NaN/Inf; clean before use",
    "Flow Packets/s": "Contains Inf; clean before use",
}.items():
    if c in df.columns:
        optional_careful[c] = reason

rows = [{"column": k, "action": "EXCLUDE", "reason": v} for k, v in exclude.items()]
rows += [{"column": k, "action": "CLEAN_BEFORE_USE", "reason": v} for k, v in optional_careful.items()]
leakage_report = pd.DataFrame(rows)
print(leakage_report.to_string(index=False))
leakage_report.to_csv(OUTPUT_DIR / "leakage_and_exclusions.csv", index=False)

candidate_features = [
    c for c in feature_num_cols
    if c not in exclude and c not in const_cols
]
print(f"\nCandidate numeric training features: {len(candidate_features)}")
(OUTPUT_DIR / "candidate_features.json").write_text(
    json.dumps({"features": candidate_features}, indent=2), encoding="utf-8"
)


### Leakage conclusions (critical for SOC ML)

**Exclude:** `Label`, `is_attack`, `source_file`, any IPs/ports/timestamps/flow IDs if present, all constant columns.  
**Clean before use:** `Flow Bytes/s`, `Flow Packets/s`.

**Production caveat:** CICFlowMeter-style features ≠ Wazuh alert JSON ≠ Suricata EVE fields.  
A CIC-trained model must not score raw SIEM events without an explicit **feature mapper**.


## 8. Recommended cleaning pipeline (next step)

1. Strip columns; normalize Label  
2. Drop duplicate feature rows  
3. Drop constant columns  
4. Inf→NaN on rate features; impute/drop  
5. Stratified split (consider blocking by `source_file`)  
6. Scale skewed numerics  
7. Handle imbalance without leaking test labels  

Artifacts: `outputs/eda/`.


## 9. Future FastAPI ↔ SOC integration (modular contract)

ML stays independent of Wazuh/Suricata configs.

### Request (draft)

```json
{
  "request_id": "uuid",
  "source": "wazuh|suricata|manual",
  "timestamp": "ISO-8601",
  "features": {"<mapped_feature>": 0.0},
  "context": {
    "agent_name": "suricata-ids",
    "src_ip": "192.168.221.30",
    "dst_ip": "192.168.221.136",
    "suricata_sid": 1000001,
    "wazuh_rule_id": "100101"
  }
}
```

### Response (draft)

```json
{
  "request_id": "uuid",
  "model_version": "cic_ids2017_baseline_v0",
  "prediction": "ATTACK",
  "label_proba": {"BENIGN": 0.12, "ATTACK": 0.88},
  "top_class": "DoS Hulk",
  "risk_score": 0.88,
  "explanations": [{"feature": "Flow Packets/s", "contribution": 0.21}],
  "notes": "Mapped SIEM fields; missing features imputed"
}
```

| Component | Role |
|-----------|------|
| Suricata / Wazuh | Unchanged detection & alerting |
| Feature mapper | EVE/Wazuh → model vector |
| FastAPI ML service | Score + risk |
| Wazuh (future) | Optional enrichment consumer |

Lab IPs: SOC `192.168.221.138`, IDS `192.168.221.136`, Kali `192.168.221.30`.


## 10. EDA summary

| Topic | Finding |
|-------|---------|
| Scale | ~2.83M flows / 8 CSVs when full CIC set is present |
| Quality | NaN/Inf in rates; ~10.9% dups; constant cols |
| Imbalance | ~80% benign; ultra-rare classes exist |
| Separability | Duration/rate/size features look informative |
| Leakage | Exclude labels, `source_file`, IDs/IPs/timestamps |
| SOC path | Keep ML modular; map SIEM events later via FastAPI |

**Next:** preprocessing + baseline models — still without modifying the live SOC stack.


## Appendix — Original notebook lineage

Original `01_eda.ipynb` cells (load + quality) are preserved conceptually in Sections 1–2 with hardening (multi-path load, exports, binary label). Keep the original file untouched as historical reference.
